<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 10


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс Invoice в C#, который будет представлять информацию о
фактурах за поставленные товары или оказанные услуги. На основе этого класса
разработать 2-3 производных класса, демонстрирующих принципы наследования и
полиморфизма. В каждом из классов должны быть реализованы новые атрибуты и
методы, а также переопределены некоторые методы базового класса для
демонстрации полиморфизма.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) создайте явную реализации интерфейса и управление зависимостями 


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [1]:
using System;
using System.Collections.Generic;

public interface IPrintable
{
    void PrintDetails();
}

public interface ILoggable
{
    void PrintDetails();
}

public interface ITaxService
{
    decimal CalculateTax(decimal amount);
}

public class StandardTaxService : ITaxService
{
    public decimal CalculateTax(decimal amount)
    {
        return amount * 0.20m;
    }
}

public class LineItem
{
    public string Name { get; set; }
    public decimal Price { get; set; }
    public int Quantity { get; set; }

    public LineItem(string name, decimal price, int quantity)
    {
        Name = name;
        Price = price;
        Quantity = quantity;
    }
}

public class Invoice : IPrintable, ILoggable
{
    public string InvoiceNumber { get; set; }
    public DateTime IssuedDate { get; set; }
    public decimal TotalAmount { get; set; }
    public List<LineItem> Items { get; set; }
    public ITaxService TaxService { get; set; }
    public string Currency { get; set; }
    public bool IsClosed { get; set; }

    public Invoice(string invoiceNumber, DateTime issuedDate, ITaxService taxService, string currency, bool isClosed)
    {
        InvoiceNumber = invoiceNumber;
        IssuedDate = issuedDate;
        TotalAmount = 0m;
        Items = new List<LineItem>();
        TaxService = taxService;
        Currency = currency;
        IsClosed = isClosed;
    }

    public virtual decimal CalculateTotal()
    {
        decimal sum = 0;
        foreach (var item in Items)
        {
            sum += item.Price * item.Quantity;
        }
        decimal tax = TaxService.CalculateTax(sum);
        TotalAmount = sum + tax;
        return TotalAmount;
    }

    public virtual void AddLine(LineItem lineItem)
    {
        Items.Add(lineItem);
        CalculateTotal();
        Console.WriteLine($"Позиция '{lineItem.Name}' добавлена в счет {InvoiceNumber}.");
    }

    public virtual void RemoveLine(LineItem lineItem)
    {
        if (Items.Contains(lineItem))
        {
            Items.Remove(lineItem);
            CalculateTotal();
            Console.WriteLine($"Позиция '{lineItem.Name}' удалена из счета {InvoiceNumber}.");
        }
    }

    public void CloseInvoice()
    {
        IsClosed = true;
        Console.WriteLine($"Счет {InvoiceNumber} закрыт.");
    }

    public void ApplyDiscount(decimal amount)
    {
        if (amount > 0 && amount <= TotalAmount)
        {
            TotalAmount -= amount;
            Console.WriteLine($"Применена скидка {amount} {Currency}. Итоговая сумма: {TotalAmount} {Currency}");
        }
    }

    void IPrintable.PrintDetails()
    {
        Console.WriteLine($"[IPrintable] Счет № {InvoiceNumber} от {IssuedDate.ToShortDateString()}, Сумма: {TotalAmount} {Currency}");
    }

    void ILoggable.PrintDetails()
    {
        Console.WriteLine($"[ILoggable] LOG: Invoice {InvoiceNumber}, Closed: {IsClosed}, Items count: {Items.Count}");
    }
}

public class GoodsInvoice : Invoice
{
    public DateTime SupplyDate { get; set; }
    public string WarehouseLocation { get; set; }
    public decimal ShippingCost { get; set; }

    public GoodsInvoice(string invoiceNumber, DateTime issuedDate, ITaxService taxService, string currency, bool isClosed,
                        DateTime supplyDate, string warehouseLocation, decimal shippingCost)
        : base(invoiceNumber, issuedDate, taxService, currency, isClosed)
    {
        SupplyDate = supplyDate;
        WarehouseLocation = warehouseLocation;
        ShippingCost = shippingCost;
    }

    public override void AddLine(LineItem lineItem)
    {
        base.AddLine(lineItem);
        Console.WriteLine($"Товар отправлен со склада '{WarehouseLocation}'. Дата поставки: {SupplyDate.ToShortDateString()}.");
    }

    public void ScheduleDelivery()
    {
        Console.WriteLine($"Доставка товаров по счету {InvoiceNumber} запланирована на {SupplyDate.ToShortDateString()}.");
    }

    public void UpdateWarehouse(string newLocation)
    {
        WarehouseLocation = newLocation;
        Console.WriteLine($"Склад изменен на: {WarehouseLocation}");
    }
}

public class ServiceInvoice : Invoice
{
    public DateTime ServiceDate { get; set; }
    public string SpecialistName { get; set; }
    public int ServiceHours { get; set; }

    public ServiceInvoice(string invoiceNumber, DateTime issuedDate, ITaxService taxService, string currency, bool isClosed,
                          DateTime serviceDate, string specialistName, int serviceHours)
        : base(invoiceNumber, issuedDate, taxService, currency, isClosed)
    {
        ServiceDate = serviceDate;
        SpecialistName = specialistName;
        ServiceHours = serviceHours;
    }

    public override void RemoveLine(LineItem lineItem)
    {
        base.RemoveLine(lineItem);
        Console.WriteLine($"Услуга '{lineItem.Name}' аннулирована специалистом {SpecialistName}. Причина: отказ клиента или перенос даты.");
    }

    public void AssignSpecialist(string newSpecialist)
    {
        SpecialistName = newSpecialist;
        Console.WriteLine($"Назначен новый специалист: {SpecialistName}");
    }

    public void LogServiceHours()
    {
        Console.WriteLine($"Затрачено часов на оказание услуг: {ServiceHours} ч.");
    }
}

public class CombinedInvoice : Invoice
{
    public bool ReturnAllowed { get; set; }
    public decimal ReturnFeeRate { get; set; }
    public int ReturnWindowDays { get; set; }

    public CombinedInvoice(string invoiceNumber, DateTime issuedDate, ITaxService taxService, string currency, bool isClosed,
                           bool returnAllowed, decimal returnFeeRate, int returnWindowDays)
        : base(invoiceNumber, issuedDate, taxService, currency, isClosed)
    {
        ReturnAllowed = returnAllowed;
        ReturnFeeRate = returnFeeRate;
        ReturnWindowDays = returnWindowDays;
    }

    public override decimal CalculateTotal()
    {
        decimal baseSum = base.CalculateTotal();
        if (ReturnAllowed)
        {
            TotalAmount = baseSum + (baseSum * ReturnFeeRate);
        }
        return TotalAmount;
    }

    public void CheckReturnPolicy()
    {
        Console.WriteLine(ReturnAllowed ? $"Возврат разрешен в течение {ReturnWindowDays} дней с комиссией {ReturnFeeRate * 100}%." : "Возврат не предусмотрен.");
    }

    public void ExtendReturnWindow(int extraDays)
    {
        ReturnWindowDays += extraDays;
        Console.WriteLine($"Срок возврата продлен. Всего дней: {ReturnWindowDays}");
    }
}

ITaxService taxService = new StandardTaxService();
LineItem item1 = new LineItem("Серверное оборудование", 120000m, 1);
LineItem item2 = new LineItem("Настройка ПО", 15000m, 2);

GoodsInvoice goodsInvoice = new GoodsInvoice("INV-001", DateTime.Now, taxService, "RUB", false, DateTime.Now.AddDays(3), "Тюмень-Логистик", 5000m);
goodsInvoice.AddLine(item1);
((IPrintable)goodsInvoice).PrintDetails();
Console.WriteLine();

ServiceInvoice serviceInvoice = new ServiceInvoice("INV-002", DateTime.Now, taxService, "RUB", false, DateTime.Now, "Артём", 10);
serviceInvoice.AddLine(item2);
serviceInvoice.RemoveLine(item2);
((ILoggable)serviceInvoice).PrintDetails();
Console.WriteLine();

CombinedInvoice combinedInvoice = new CombinedInvoice("INV-003", DateTime.Now, taxService, "RUB", false, true, 0.05m, 14);
combinedInvoice.AddLine(item1);
combinedInvoice.CheckReturnPolicy();
Console.WriteLine($"Итоговая сумма комбинированного счета: {combinedInvoice.CalculateTotal()} RUB");

Позиция 'Серверное оборудование' добавлена в счет INV-001.
Товар отправлен со склада 'Тюмень-Логистик'. Дата поставки: 11/7/2026.
[IPrintable] Счет № INV-001 от 11/4/2026, Сумма: 144000.00 RUB

Позиция 'Настройка ПО' добавлена в счет INV-002.
Позиция 'Настройка ПО' удалена из счета INV-002.
Услуга 'Настройка ПО' аннулирована специалистом Артём. Причина: отказ клиента или перенос даты.
[ILoggable] LOG: Invoice INV-002, Closed: False, Items count: 0

Позиция 'Серверное оборудование' добавлена в счет INV-003.
Возврат разрешен в течение 14 дней с комиссией 5.00%.
Итоговая сумма комбинированного счета: 151200.0000 RUB
